# Chapter 12: Optimization

## Unconstrained Optimization

### Python Implementation: Unconstrained Optimization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import time

# Define a test function: Rosenbrock function
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

# Gradient of Rosenbrock function
def rosenbrock_grad(x):
    dfdx0 = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2)
    dfdx1 = 200 * (x[1] - x[0]**2)
    return np.array([dfdx0, dfdx1])

# Hessian of Rosenbrock function
def rosenbrock_hess(x):
    h00 = 2 + 800 * x[0]**2 - 400 * (x[1] - x[0]**2)
    h01 = -400 * x[0]
    h11 = 200
    return np.array([[h00, h01], [h01, h11]])

# Example: Comparing different methods
x0 = np.array([-1.2, 1.0])
methods_to_test = {
    'BFGS': {'method': 'BFGS', 'jac': rosenbrock_grad},
    'CG': {'method': 'CG', 'jac': rosenbrock_grad},
    'Newton-CG': {'method': 'Newton-CG', 'jac': rosenbrock_grad,
                  'hess': rosenbrock_hess},
    'Powell': {'method': 'Powell'},
    'L-BFGS-B': {'method': 'L-BFGS-B', 'jac': rosenbrock_grad},
}

results = {}
print("=" * 80)
print("Optimization Results on Rosenbrock Function")
print("=" * 80)

for method_name, method_kwargs in methods_to_test.items():
    start = time.time()
    result = minimize(rosenbrock, x0, **method_kwargs)
    elapsed = time.time() - start
    results[method_name] = result

    print(f"\n{method_name}:")
    print(f"  Optimal point: {result.x}")
    print(f"  Function value: {result.fun:.6e}")
    print(f"  Iterations: {result.nit}")
    print(f"  Function evals: {result.nfev}")
    print(f"  Time: {elapsed:.6f} s")
    print(f"  Success: {result.success}")

# Plot convergence for methods with history
print("\n" + "=" * 80)

# Visualize convergence behavior for Rosenbrock
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Function values vs iterations (symbolic)
ax = axes[0]
methods = list(results.keys())
iters = [results[m].nit for m in methods]
fevals = [results[m].nfev for m in methods]
colors = plt.cm.tab10(np.linspace(0, 1, len(methods)))

ax.bar(range(len(methods)), iters, alpha=0.7, label='Iterations', color=colors)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.set_ylabel('Number of Iterations')
ax.set_title('Convergence Comparison')
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Function evaluations
ax = axes[1]
ax.bar(range(len(methods)), fevals, alpha=0.7, color=colors)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.set_ylabel('Function Evaluations')
ax.set_title('Computational Cost Comparison')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Example: Least Squares Fitting

In [ ]:
import numpy as np
from scipy.optimize import least_squares, minimize
import matplotlib.pyplot as plt

# Generate synthetic data
np.random.seed(42)
x = np.linspace(0, 10, 50)
y_true = 2 + 3*x + 0.5*x**2
y_data = y_true + np.random.normal(0, 2, len(x))

# Define the model and residuals
def poly_model(params, x):
    a, b, c = params
    return a + b*x + c*x**2

def residuals(params, x, y):
    return y - poly_model(params, x)

def objective(params, x, y):
    res = residuals(params, x, y)
    return np.sum(res**2)

# Initial guess and optimization
params_init = np.array([0.0, 1.0, 0.0])
result_ls = least_squares(residuals, params_init, args=(x, y_data))
result_min = minimize(objective, params_init, args=(x, y_data),
                      method='BFGS')

print("Least Squares Fitting Results:")
print(f"True params: a=2.0, b=3.0, c=0.5")
print(f"Fitted params: a={result_ls.x[0]:.4f}, b={result_ls.x[1]:.4f}, "
      f"c={result_ls.x[2]:.4f}")
print(f"Residual norm: {result_ls.cost:.6e}")

# Plotting
y_fitted = poly_model(result_ls.x, x)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(x, y_data, label='Data', alpha=0.6)
ax.plot(x, y_fitted, 'r-', label='Fitted polynomial', linewidth=2)
ax.plot(x, y_true, 'g--', label='True polynomial', linewidth=2)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
ax.set_title('Polynomial Fitting')
ax.grid(True, alpha=0.3)

# Residuals
ax = axes[1]
residuals_vals = y_data - y_fitted
ax.bar(range(len(residuals_vals)), residuals_vals, alpha=0.7)
ax.axhline(0, color='r', linestyle='--', linewidth=2)
ax.set_ylabel('Residuals')
ax.set_title('Residual Distribution')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Constrained Optimization

### Python Implementation: Constrained Optimization

In [ ]:
import numpy as np
from scipy.optimize import minimize, LinearConstraint, Bounds
import matplotlib.pyplot as plt

# Define a constrained portfolio optimization problem
# Minimize portfolio variance subject to return constraint and weights sum to 1

# Generate synthetic data
np.random.seed(42)
n_assets = 5
returns = np.array([0.08, 0.10, 0.12, 0.09, 0.11])  # Expected returns
cov_matrix = np.array([
    [0.04, 0.01, 0.02, 0.005, 0.01],
    [0.01, 0.05, 0.01, 0.02, 0.015],
    [0.02, 0.01, 0.06, 0.01, 0.02],
    [0.005, 0.02, 0.01, 0.03, 0.01],
    [0.01, 0.015, 0.02, 0.01, 0.04]
])

# Objective: portfolio variance
def portfolio_variance(weights):
    return weights @ cov_matrix @ weights

# Constraint 1: portfolio return >= target_return
target_return = 0.095
def return_constraint(weights):
    return returns @ weights - target_return

# Initial guess
x0 = np.ones(n_assets) / n_assets

# Constraints: sum weights = 1, weights >= 0, weights <= 1
constraints = [
    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
    {'type': 'ineq', 'fun': return_constraint}
]

bounds = Bounds(0, 1)

# Solve
result = minimize(portfolio_variance, x0, method='SLSQP',
                 bounds=bounds, constraints=constraints)

print("Portfolio Optimization Results:")
print(f"Optimal weights: {result.x}")
print(f"Portfolio variance: {result.fun:.6f}")
print(f"Portfolio return: {returns @ result.x:.6f}")
print(f"Portfolio std dev: {np.sqrt(result.fun):.6f}")

# Remember the target-return portfolio before re-using `result`
var_target = result.fun
ret_target = returns @ result.x

# Compute efficient frontier
target_returns = np.linspace(returns.min(), returns.max(), 20)
min_variances = []
optimal_weights_list = []

for target in target_returns:
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'ineq', 'fun': lambda w: returns @ w - target}
    ]
    result = minimize(portfolio_variance, x0, method='SLSQP',
                     bounds=bounds, constraints=constraints)
    if result.success:
        min_variances.append(result.fun)
        optimal_weights_list.append(result.x)
    else:
        min_variances.append(np.nan)
        optimal_weights_list.append(np.full(n_assets, np.nan))

# Plot efficient frontier
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
min_stds = np.sqrt(np.array(min_variances))
ax.plot(min_stds, target_returns, 'b-', linewidth=2, label='Efficient Frontier')
ax.scatter(np.sqrt(var_target), ret_target, color='r', s=100,
          label='Target Return Portfolio', zorder=5)
ax.set_xlabel('Portfolio Std Dev (annual)')
ax.set_ylabel('Expected Return (annual)')
ax.set_title('Efficient Frontier')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot optimal weights for different returns
ax = axes[1]
optimal_weights_array = np.array(optimal_weights_list)
for i in range(n_assets):
    ax.plot(target_returns, optimal_weights_array[:, i], 'o-',
           label=f'Asset {i+1}', markersize=3)

ax.set_xlabel('Target Return')
ax.set_ylabel('Weight')
ax.set_title('Optimal Asset Weights vs Target Return')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Multi-dimensional Optimization and Algorithm Comparison

### Visualization: Optimization Paths

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

def rosenbrock_grad(x):
    dfdx0 = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2)
    dfdx1 = 200 * (x[1] - x[0]**2)
    return np.array([dfdx0, dfdx1])

# Track optimization paths using callback
class OptimizationTracker:
    def __init__(self):
        self.history = []

    def callback(self, xk):
        self.history.append(np.copy(xk))

x0 = np.array([-1.2, 1.0])
methods = ['BFGS', 'CG', 'Powell']
histories = {}

for method in methods:
    tracker = OptimizationTracker()
    result = minimize(rosenbrock, x0, method=method,
                     jac=rosenbrock_grad if method != 'Powell' else None,
                     callback=tracker.callback, options={'maxiter': 500})
    histories[method] = np.array(tracker.history)

# Create contour plot with optimization paths
fig, ax = plt.subplots(figsize=(12, 10))

# Contour plot
x = np.linspace(-2, 2, 200)
y = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(x, y)
Z = rosenbrock(np.array([X, Y]))

levels = np.logspace(-1, 5, 40)
contour = ax.contour(X, Y, Z, levels=levels, cmap='viridis', alpha=0.7)
ax.clabel(contour, inline=True, fontsize=8)

# Plot paths
colors = {'BFGS': 'red', 'CG': 'blue', 'Powell': 'green'}
for method, history in histories.items():
    ax.plot(history[:, 0], history[:, 1], 'o-', color=colors[method],
           markersize=3, linewidth=1.5, alpha=0.7, label=method)
    ax.plot(history[0, 0], history[0, 1], 'o', color=colors[method],
           markersize=10, markeredgecolor='black', markeredgewidth=2)
    ax.plot(history[-1, 0], history[-1, 1], '*', color=colors[method],
           markersize=20, markeredgecolor='black', markeredgewidth=1)

ax.plot(1, 1, 'r*', markersize=25, markeredgecolor='black',
       markeredgewidth=2, label='Global optimum')

ax.set_xlim(-2, 2)
ax.set_ylim(-1, 3)
ax.set_xlabel('$x_0$', fontsize=12)
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_title('Optimization Paths on Rosenbrock Function', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Linear Programming and Quadratic Programming

### Quadratic Programming

In [ ]:
import numpy as np
from scipy.optimize import linprog, minimize
import matplotlib.pyplot as plt

# Data
returns = np.array([0.08, 0.10, 0.12, 0.09, 0.11])
n_assets = len(returns)
cov_matrix = np.array([
    [0.04, 0.01, 0.02, 0.005, 0.01],
    [0.01, 0.05, 0.01, 0.02, 0.015],
    [0.02, 0.01, 0.06, 0.01, 0.02],
    [0.005, 0.02, 0.01, 0.03, 0.01],
    [0.01, 0.015, 0.02, 0.01, 0.04]
])

# Problem 1: LP - maximize return with fixed risk
# min -returns^T x = max returns^T x
# subject to: variance <= sigma_max^2, sum(x) = 1, x >= 0

# For LP, convert variance constraint to linear approximation
# (quadratic => cannot use LP directly, so use QP instead)

# Problem 2: QP - minimize variance with return constraint
Q = 2 * cov_matrix  # Hessian for 0.5 x^T Q x
c = np.zeros(n_assets)
target_return = 0.095

# Constraint: returns^T x >= target_return (linear)
# Constraint: sum(x) = 1 (linear)
# Bounds: 0 <= x <= 1 (linear)

# Using scipy minimize for QP (since linprog only handles LP)
def qp_objective(x):
    return 0.5 * x @ cov_matrix @ x

constraints = [
    {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
    {'type': 'ineq', 'fun': lambda x: returns @ x - target_return}
]

x0 = np.ones(n_assets) / n_assets
bounds = [(0, 1) for _ in range(n_assets)]

result = minimize(qp_objective, x0, method='SLSQP',
                 bounds=bounds, constraints=constraints)

print("QP Solution:")
print(f"Optimal weights: {result.x}")
print(f"Portfolio variance: {2 * result.fun:.6f}")  # objective is x' Sigma x / 2
print(f"Portfolio return: {returns @ result.x:.6f}")

# Alternative: Using diet problem as LP example
print("\n" + "=" * 60)
print("Linear Programming Example: Diet Problem")
print("=" * 60)

# Foods: chicken, beef, fish, rice
# Nutrients: protein, fat, carbs
# Nutritional content and cost
foods = ['Chicken', 'Beef', 'Fish', 'Rice']
cost = np.array([2.0, 3.0, 4.0, 0.5])  # $ per serving

nutrition = np.array([
    [26, 15, 20, 3],   # Protein (g per serving)
    [5, 15, 10, 1],    # Fat (g per serving)
    [0, 0, 0, 45],     # Carbs (g per serving)
])

# Daily requirements: protein >= 50g, fat <= 50g, carbs >= 100g
req_min_protein = 50
req_max_fat = 50
req_min_carbs = 100

# LP: min cost^T x subject to constraints
# Rewrite as: min c^T x, subject to A_ub @ x <= b_ub, A_eq @ x = b_eq, x >= 0

A_ub = np.array([
    -nutrition[0, :],  # -protein <= -50  <=>  protein >= 50
    nutrition[1, :],   # fat <= 50
    -nutrition[2, :]   # -carbs <= -100  <=>  carbs >= 100
])

b_ub = np.array([-req_min_protein, req_max_fat, -req_min_carbs])
bounds_diet = [(0, None) for _ in range(4)]

result_diet = linprog(cost, A_ub=A_ub, b_ub=b_ub, bounds=bounds_diet,
                     method='highs')

print("\nDiet Problem Solution:")
servings_per_food = {f: float(round(s, 3)) for f, s in zip(foods, result_diet.x)}
print(f"Optimal servings: {servings_per_food}")
print(f"Minimum daily cost: ${result_diet.fun:.2f}")

# Verify constraints
print(f"\nNutrient content:")
servings = result_diet.x
print(f"Protein: {nutrition[0, :] @ servings:.1f}g (required >= {req_min_protein}g)")
print(f"Fat: {nutrition[1, :] @ servings:.1f}g (required <= {req_max_fat}g)")
print(f"Carbs: {nutrition[2, :] @ servings:.1f}g (required >= {req_min_carbs}g)")

## Global Optimization with Non-convex Problems

### Python Implementation: Global Optimization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import (minimize, differential_evolution,
                           dual_annealing, basinhopping)

def rastrigin(x):
    n = len(x)
    return 10 * n + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))

def ackley(x):
    n = len(x)
    sum1 = np.sum(x**2)
    sum2 = np.sum(np.cos(2 * np.pi * x))
    return (-20 * np.exp(-0.2 * np.sqrt(sum1/n))
            - np.exp(sum2/n) + 20 + np.e)

# Bounds for 2D problems
bounds = [(-5.12, 5.12), (-5.12, 5.12)]

# Test on Rastrigin
print("=" * 70)
print("Global Optimization: Rastrigin Function")
print("=" * 70)

# Differential Evolution
result_de = differential_evolution(rastrigin, bounds, seed=42,
                                  maxiter=1000, atol=1e-10)
print(f"\nDifferential Evolution:")
print(f"  Best point: {result_de.x}")
print(f"  Best value: {result_de.fun:.6e}")
print(f"  Evaluations: {result_de.nfev}")

# Dual Annealing
result_da = dual_annealing(rastrigin, bounds, seed=42, maxiter=1000)
print(f"\nDual Annealing:")
print(f"  Best point: {result_da.x}")
print(f"  Best value: {result_da.fun:.6e}")
print(f"  Evaluations: {result_da.nfev}")

# Basin-Hopping
minimizer_kwargs = {'method': 'BFGS'}
result_bh = basinhopping(rastrigin, np.array([2.0, 2.0]),
                        minimizer_kwargs=minimizer_kwargs,
                        niter=100, seed=42)
print(f"\nBasin-Hopping:")
print(f"  Best point: {result_bh.x}")
print(f"  Best value: {result_bh.fun:.6e}")

# Visualize results on 2D Rastrigin
# NOTE: rastrigin() sums over its whole argument, so calling it on a
# meshgrid stack collapses everything to a scalar. Evaluate the 2D
# version explicitly on the grid instead.
def rastrigin_grid(X, Y):
    return (20 + X**2 - 10 * np.cos(2 * np.pi * X)
               + Y**2 - 10 * np.cos(2 * np.pi * Y))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full function
x = np.linspace(-5.12, 5.12, 200)
y = np.linspace(-5.12, 5.12, 200)
X, Y = np.meshgrid(x, y)
Z = rastrigin_grid(X, Y)

ax = axes[0]
contour = ax.contourf(X, Y, Z, levels=50, cmap='viridis')
ax.plot([0], [0], 'r*', markersize=20, label='Global optimum')
ax.plot(result_de.x[0], result_de.x[1], 'r+', markersize=15,
       markeredgewidth=2, label=f'DE: f={result_de.fun:.3e}')
ax.plot(result_da.x[0], result_da.x[1], 'b+', markersize=15,
       markeredgewidth=2, label=f'DA: f={result_da.fun:.3e}')
ax.set_xlabel('$x_0$')
ax.set_ylabel('$x_1$')
ax.set_title('Rastrigin Function: Global View')
ax.legend()
plt.colorbar(contour, ax=ax, label='$f(x_0, x_1)$')

# Zoomed view
x_zoom = np.linspace(-2, 2, 200)
y_zoom = np.linspace(-2, 2, 200)
X_zoom, Y_zoom = np.meshgrid(x_zoom, y_zoom)
Z_zoom = rastrigin_grid(X_zoom, Y_zoom)

ax = axes[1]
contour = ax.contourf(X_zoom, Y_zoom, Z_zoom, levels=30, cmap='viridis')
ax.plot([0], [0], 'r*', markersize=20, label='Global optimum')
ax.set_xlabel('$x_0$')
ax.set_ylabel('$x_1$')
ax.set_title('Rastrigin Function: Zoomed Near Optimum')
ax.legend()
plt.colorbar(contour, ax=ax, label='$f(x_0, x_1)$')

plt.tight_layout()
plt.show()

# Performance comparison on different test functions
print("\n" + "=" * 70)
print("Performance Comparison")
print("=" * 70)

test_functions = {
    'Rastrigin': rastrigin,
    'Ackley': ackley,
}

for func_name, func in test_functions.items():
    print(f"\n{func_name}:")

    result_de = differential_evolution(func, bounds, seed=42, maxiter=1000)
    result_da = dual_annealing(func, bounds, seed=42, maxiter=1000)

    print(f"  DE: f = {result_de.fun:.6e}, evals = {result_de.nfev}")
    print(f"  DA: f = {result_da.fun:.6e}, evals = {result_da.nfev}")

## Sensitivity and Uncertainty Analysis

### Python Implementation: Sensitivity Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model: Beam deflection
def beam_deflection(p):
    # p = [F, L, E, I]
    F, L, E, I = p
    return F * L**3 / (3 * E * I)  # y_max = F L^3 / (3 E I)

param_bounds = {
    'F': (100, 200),      # Force (N)
    'L': (0.5, 1.5),      # Length (m)
    'E': (200e9, 220e9),  # Young's modulus (Pa)
    'I': (1e-6, 2e-6),    # Moment (m^4)
}

param_names = ['Force (F)', 'Length (L)', 'Modulus (E)', 'Moment (I)']
n_params = 4

# Morris Method
print("=" * 70)
print("MORRIS SENSITIVITY ANALYSIS")
print("=" * 70)

n_trajectories = 20
delta = 0.2
mu_star = np.zeros(n_params)
sigma = np.zeros(n_params)
elementary_effects = [[] for _ in range(n_params)]

np.random.seed(42)

for traj in range(n_trajectories):
    p_norm = np.random.rand(n_params)
    param_order = np.random.permutation(n_params)

    p_current = np.zeros(n_params)
    for i, name in enumerate(['F', 'L', 'E', 'I']):
        p_min, p_max = param_bounds[name]
        p_current[i] = p_min + p_norm[i] * (p_max - p_min)

    f_current = beam_deflection(p_current)

    for i in param_order:
        p_perturbed = p_current.copy()
        name = ['F', 'L', 'E', 'I'][i]
        p_min, p_max = param_bounds[name]
        param_range = p_max - p_min

        if p_current[i] + delta * param_range <= p_max:
            p_perturbed[i] += delta * param_range
        else:
            p_perturbed[i] -= delta * param_range

        f_perturbed = beam_deflection(p_perturbed)
        # Elementary effect w.r.t. the normalized parameter (step = delta)
        ee = abs(f_perturbed - f_current) / delta
        elementary_effects[i].append(ee)

        p_current = p_perturbed
        f_current = f_perturbed

for i in range(n_params):
    mu_star[i] = np.mean(elementary_effects[i])
    sigma[i] = np.std(elementary_effects[i])

print("\nMorris Sensitivity Ranking:")
print(f"{'Parameter':<20} {'mu*':<15} {'sigma':<15}")
print("-" * 50)
for i, name in enumerate(param_names):
    print(f"{name:<20} {mu_star[i]:<15.6e} {sigma[i]:<15.6e}")

# Plot Morris
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = ['red' if mu_star[i] > np.mean(mu_star)*1.5 else 'blue'
         for i in range(n_params)]
ax.scatter(mu_star, sigma, s=200, alpha=0.6, c=colors)
for i, name in enumerate(param_names):
    ax.annotate(name, (mu_star[i], sigma[i]),
               xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('$\mu^*$ (Mean Absolute Effect)', fontsize=12)
ax.set_ylabel('$\sigma$ (Standard Deviation)', fontsize=12)
ax.set_title('Morris Method: Parameter Importance', fontsize=12)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.boxplot([elementary_effects[i] for i in range(n_params)],
          tick_labels=param_names, patch_artist=True)
ax.set_ylabel('|Elementary Effect|', fontsize=12)
ax.set_title('Distribution of Elementary Effects', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.show()

# Monte Carlo Sobol Indices
print("\n" + "=" * 70)
print("SOBOL INDICES (MONTE CARLO)")
print("=" * 70)

n_samples = 10000
A = np.zeros((n_samples, n_params))
B = np.zeros((n_samples, n_params))

for i, name in enumerate(['F', 'L', 'E', 'I']):
    p_min, p_max = param_bounds[name]
    A[:, i] = np.random.uniform(p_min, p_max, n_samples)
    B[:, i] = np.random.uniform(p_min, p_max, n_samples)

f_A = np.array([beam_deflection(A[k, :]) for k in range(n_samples)])
f_B = np.array([beam_deflection(B[k, :]) for k in range(n_samples)])

f_mean = np.mean(np.concatenate([f_A, f_B]))
f_var = np.var(np.concatenate([f_A, f_B]))

sobol_first = np.zeros(n_params)
for i in range(n_params):
    C_i = A.copy()
    C_i[:, i] = B[:, i]
    f_C_i = np.array([beam_deflection(C_i[k, :]) for k in range(n_samples)])

    numerator = np.sum(f_B * (f_C_i - f_A)) / n_samples
    sobol_first[i] = numerator / f_var

print("\nFirst-Order Sobol Indices:")
print(f"{'Parameter':<20} {'S_i':<15} {'% Variance':<15}")
print("-" * 50)
for i, name in enumerate(param_names):
    percent = sobol_first[i] * 100
    print(f"{name:<20} {sobol_first[i]:<15.6f} {percent:<15.2f}%")

# Plot Sobol
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(n_params), sobol_first,
             color=['red', 'blue', 'green', 'orange'], alpha=0.7)
ax.set_xticks(range(n_params))
ax.set_xticklabels(param_names, rotation=15, ha='right')
ax.set_ylabel('First-Order Sobol Index', fontsize=12)
ax.set_title('Global Sensitivity: Sobol Indices', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{sobol_first[i]:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## Applications in Engineering and Machine Learning

### Python: Engineering Case Study

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Cantilever beam: minimize volume subject to deflection and stress
# Design variables: width w, height h
# Constraints: max deflection <= 0.01 L, max stress <= sigma_yield

L = 1.0      # Length (m)
F = 1000.0   # Load (N)
E = 210e9    # Young's modulus (Pa)
sigma_yield = 250e6  # Yield stress (Pa)

# Objective: minimize volume V = L * w * h
def objective(x):
    w, h = x
    return L * w * h

# Constraint 1: deflection <= 0.01 L
# delta_max = 4 * F * L^3 / (E * w * h^3)
def constraint_deflection(x):
    w, h = x
    delta_max_limit = 0.01 * L
    delta_max = 4 * F * L**3 / (E * w * h**3)
    return delta_max_limit - delta_max

# Constraint 2: stress <= sigma_yield
# sigma_max = 6 * F * L / (w * h^2)
def constraint_stress(x):
    w, h = x
    sigma_max = 6 * F * L / (w * h**2)
    return sigma_yield - sigma_max

# Bounds: w, h > 0 (reasonable upper bounds)
bounds = [(0.001, 0.1), (0.001, 0.2)]

# Initial guess
x0 = np.array([0.05, 0.1])

constraints = [
    {'type': 'ineq', 'fun': constraint_deflection},
    {'type': 'ineq', 'fun': constraint_stress}
]

result = minimize(objective, x0, method='SLSQP', bounds=bounds,
                 constraints=constraints)

print("Beam Optimization Results:")
print(f"Optimal width: {result.x[0]:.6f} m")
print(f"Optimal height: {result.x[1]:.6f} m")
print(f"Minimum volume: {result.fun:.6e} m^3")

w_opt, h_opt = result.x
delta_opt = 4 * F * L**3 / (E * w_opt * h_opt**3)
sigma_opt = 6 * F * L / (w_opt * h_opt**2)

print(f"\nConstraint satisfaction:")
print(f"Max deflection: {delta_opt:.6f} m ({delta_opt/L*100:.4f}% of L)")
print(f"Max stress: {sigma_opt/1e6:.2f} MPa (limit: {sigma_yield/1e6:.2f} MPa)")

# Sensitivity analysis: how does optimal volume change with load F?
F_range = np.linspace(500, 2000, 20)
volumes = []

for F_test in F_range:
    result_test = minimize(objective, x0, method='SLSQP',
                          bounds=bounds,
                          constraints=[
        {'type': 'ineq', 'fun': lambda x: 0.01*L - 4*F_test*L**3/(E*x[0]*x[1]**3)},
        {'type': 'ineq', 'fun': lambda x: sigma_yield - 6*F_test*L/(x[0]*x[1]**2)}
                          ])
    if result_test.success:
        volumes.append(result_test.fun)
    else:
        volumes.append(np.nan)

# Plot sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(F_range, volumes, 'b-o', linewidth=2)
ax.axvline(F, color='r', linestyle='--', label=f'Current load: {F} N')
ax.set_xlabel('Load (N)', fontsize=12)
ax.set_ylabel('Optimal Volume (m$^3$)', fontsize=12)
ax.set_title('Sensitivity of Optimal Volume to Load', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Parametric study: volume vs stress limit
sigma_limits = np.linspace(200e6, 400e6, 20)
volumes_stress = []

for sigma_lim in sigma_limits:
    result_test = minimize(objective, x0, method='SLSQP',
                          bounds=bounds,
                          constraints=[
        {'type': 'ineq', 'fun': lambda x: 0.01*L - 4*F*L**3/(E*x[0]*x[1]**3)},
        {'type': 'ineq', 'fun': lambda x: sigma_lim - 6*F*L/(x[0]*x[1]**2)}
                          ])
    if result_test.success:
        volumes_stress.append(result_test.fun)
    else:
        volumes_stress.append(np.nan)

ax = axes[1]
ax.plot(sigma_limits/1e6, np.array(volumes_stress)*1e6, 'g-o', linewidth=2)
ax.axvline(sigma_yield/1e6, color='r', linestyle='--',
          label=f'Material limit: {sigma_yield/1e6:.0f} MPa')
ax.set_xlabel('Stress Limit (MPa)', fontsize=12)
ax.set_ylabel('Optimal Volume ($10^{-6}$ m$^3$)', fontsize=12)
ax.set_title('Effect of Stress Constraint on Design', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Advanced Topics in Numerical Optimization

### Preconditioning and Scaling

#### Common Preconditioners

In [ ]:
import numpy as np
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import cg, spilu, LinearOperator

# Create an ill-conditioned symmetric positive definite system
n = 100
rng = np.random.default_rng(0)
eigenvalues = np.logspace(0, 4, n)  # condition number 10^4
Q, _ = np.linalg.qr(rng.standard_normal((n, n)))
A = Q @ np.diag(eigenvalues) @ Q.T
b = rng.standard_normal(n)

def cg_iterations(A, b, M=None):
    """Run CG and return (solution, iteration count)."""
    count = {'k': 0}
    def cb(xk):
        count['k'] += 1
    x, info = cg(A, b, M=M, maxiter=10000, callback=cb)
    return x, count['k']

# Unpreconditioned CG
x_plain, iters_plain = cg_iterations(A, b)

# Diagonal (Jacobi) preconditioning: M approximates A^{-1}
M_jacobi = np.diag(1.0 / np.diag(A))
x_jac, iters_jac = cg_iterations(A, b, M=M_jacobi)

# Incomplete LU preconditioning
ilu = spilu(csc_matrix(A), drop_tol=1e-4, fill_factor=10)
M_ilu = LinearOperator((n, n), matvec=ilu.solve)
x_ilu, iters_ilu = cg_iterations(A, b, M=M_ilu)

kappa = eigenvalues.max() / eigenvalues.min()
print(f"Condition number of A: {kappa:.0e}")
print(f"Unpreconditioned CG:      {iters_plain} iterations")
print(f"Jacobi-preconditioned CG: {iters_jac} iterations")
print(f"ILU-preconditioned CG:    {iters_ilu} iterations")

### Proximal Methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sparse regression: min || y - X * theta ||^2 + lambda || theta ||_1

np.random.seed(42)
n_samples, n_features = 100, 50
X = np.random.randn(n_samples, n_features)
theta_true = np.zeros(n_features)
theta_true[[5, 15, 25, 35]] = [1.0, -1.5, 2.0, -0.5]
y = X @ theta_true + np.random.normal(0, 0.1, n_samples)

# Soft thresholding (proximal operator of L1)
def soft_threshold(x, threshold):
    return np.sign(x) * np.maximum(np.abs(x) - threshold, 0)

# Proximal gradient descent
def proximal_gradient_descent(X, y, lam, alpha=None, max_iters=1000,
                              tol=1e-6):
    n_features = X.shape[1]
    if alpha is None:
        # Largest stable step: 1/L with L = lambda_max(X^T X)
        alpha = 1.0 / np.linalg.norm(X, 2)**2
    theta = np.zeros(n_features)
    losses = []

    for k in range(max_iters):
        # Gradient of the smooth part 0.5 * ||y - X theta||^2
        residual = y - X @ theta
        gradient = -X.T @ residual

        # Proximal gradient step
        theta_new = soft_threshold(theta - alpha * gradient, alpha * lam)

        # Objective: 0.5 * ||y - X theta||^2 + lam * ||theta||_1
        loss = (0.5 * np.sum((y - X @ theta_new)**2)
                + lam * np.sum(np.abs(theta_new)))
        losses.append(loss)

        if np.linalg.norm(theta_new - theta) < tol:
            break

        theta = theta_new

    return theta, losses

# Solve for different lambda values
lambdas = [0.01, 0.1, 1.0, 10.0]
results = {}

for lam in lambdas:
    theta_opt, losses = proximal_gradient_descent(X, y, lam)
    results[lam] = (theta_opt, losses)
    mse = np.mean((X @ theta_opt - y)**2)
    sparsity = np.sum(theta_opt == 0)
    print(f"Lambda={lam}: MSE={mse:.6f}, Sparsity={sparsity}/{n_features}")

# Visualize solution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(np.arange(n_features), theta_true, width=0.6, alpha=0.3,
      color='black', label='True theta')
markers = {0.01: 'o', 0.1: 's', 1.0: '^', 10.0: 'v'}
for lam, (theta_opt, _) in results.items():
    ax.plot(np.arange(n_features), theta_opt, markers[lam],
           alpha=0.7, markersize=4, linestyle='none',
           label=f'lambda={lam}')
ax.set_xlabel('Parameter Index')
ax.set_ylabel('Coefficient Value')
ax.set_title('Sparse Regression: Effect of Regularization')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for lam, (_, losses) in results.items():
    ax.plot(losses, label=f'lambda={lam}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Objective Value')
ax.set_yscale('log')
ax.set_title('Convergence of Proximal Gradient')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Detailed Worked Examples

### Example 2: Nonlinear Least Squares with Robust Regularization

In [ ]:
from scipy.optimize import minimize
import numpy as np
import matplotlib.pyplot as plt

# Data with outliers
np.random.seed(7)
x_data = np.linspace(0, 8, 40)
y_true = 2.0 * np.exp(-0.5 * x_data) + 0.5
y_data = y_true + np.random.normal(0, 0.1, len(x_data))
# Add outliers
y_data[[4, 12, 22, 30]] += 2.0

# Nonlinear model
def model(x, theta):
    return theta[0] * np.exp(-theta[1] * x) + theta[2]

# Huber loss
def huber_loss(residuals, delta=0.2):
    return np.sum(np.where(np.abs(residuals) <= delta,
                          0.5 * residuals**2,
                          delta * (np.abs(residuals) - 0.5*delta)))

# Robust and plain least-squares objectives
def objective_huber(theta):
    return huber_loss(y_data - model(x_data, theta))

def objective_lsq(theta):
    return 0.5 * np.sum((y_data - model(x_data, theta))**2)

# Fit with both losses
theta_init = np.array([1.0, 0.3, 0.0])
theta_huber = minimize(objective_huber, theta_init, method='BFGS').x
theta_lsq = minimize(objective_lsq, theta_init, method='BFGS').x

print(f"True parameters:  [2.0, 0.5, 0.5]")
print(f"Huber fit:        {np.round(theta_huber, 3)}")
print(f"Least-squares fit: {np.round(theta_lsq, 3)}")

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Data (with outliers)', s=60, alpha=0.6)
plt.plot(x_data, y_true, 'g-', label='True model', linewidth=2)
plt.plot(x_data, model(x_data, theta_huber), 'r--',
         label='Huber fit', linewidth=2)
plt.plot(x_data, model(x_data, theta_lsq), 'b:',
         label='Least-squares fit', linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.title('Nonlinear Regression with Robust Loss')
plt.grid(True, alpha=0.3)
plt.show()

### Example 3: Structured Optimization –- Low-Rank Matrix Recovery

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd

# Generate low-rank matrix and add noise
np.random.seed(42)
rank_true = 3
n = 20
U_true = np.random.randn(n, rank_true)
V_true = np.random.randn(n, rank_true)
X_true = U_true @ V_true.T

# Observed data with noise
A = X_true + 0.1 * np.random.randn(n, n)

# Proximal gradient descent for nuclear norm + squared error
def svd_soft_threshold(X, threshold):
    U, S, Vt = svd(X, full_matrices=False)
    S_thresholded = np.maximum(S - threshold, 0)
    return U @ np.diag(S_thresholded) @ Vt

def objective(X, A, lam):
    U, S, _ = svd(X, full_matrices=False)
    nuclear_norm = np.sum(S)
    squared_error = np.sum((X - A)**2)
    return nuclear_norm + (lam/2) * squared_error

# Proximal gradient descent
# The smooth part (lam/2)||X - A||_F^2 has Lipschitz constant lam,
# so any step alpha <= 1/lam is stable.
X_curr = A.copy()
lam = 1.0
alpha = 0.5
max_iters = 100

objectives = []
for k in range(max_iters):
    # Gradient of smooth part
    grad = lam * (X_curr - A)

    # Proximal gradient step
    X_temp = X_curr - alpha * grad
    X_new = svd_soft_threshold(X_temp, alpha)

    obj = objective(X_new, A, lam)
    objectives.append(obj)

    if np.linalg.norm(X_new - X_curr) < 1e-6:
        break

    X_curr = X_new

print(f"True rank: {rank_true}")
print(f"Recovered rank (singular values > 1e-3): {np.sum(np.linalg.svd(X_curr)[1] > 1e-3)}")
print(f"Recovery error: {np.linalg.norm(X_curr - X_true)/np.linalg.norm(X_true):.6f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(objectives)
ax.set_xlabel('Iteration')
ax.set_ylabel('Objective Value')
ax.set_title('Convergence of Nuclear Norm Minimization')
ax.grid(True, alpha=0.3)

ax = axes[1]
singular_vals = np.linalg.svd(X_curr)[1]
ax.semilogy(singular_vals, 'bo-', label='Recovered singular values')
ax.axhline(1e-3, color='r', linestyle='--', label='Threshold')
ax.set_xlabel('Index')
ax.set_ylabel('Singular Value')
ax.set_title('Singular Value Spectrum')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Example 5: Robust Principal Component Analysis (RPCA)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd

# Generate synthetic data: low-rank + sparse corruption
np.random.seed(42)
m, n = 50, 100
rank = 5

# Low-rank component
U = np.random.randn(m, rank)
V = np.random.randn(n, rank)
L_true = U @ V.T

# Sparse corruption
S_true = np.zeros((m, n))
corr_indices = np.random.choice(m*n, size=int(0.1*m*n), replace=False)
S_true.ravel()[corr_indices] = np.random.randn(len(corr_indices)) * 3

M = L_true + S_true

# RPCA via alternating direction method
def soft_threshold(X, threshold):
    return np.sign(X) * np.maximum(np.abs(X) - threshold, 0)

def svd_soft_threshold(X, threshold):
    U_s, S_s, Vt_s = svd(X, full_matrices=False)
    S_thresholded = np.maximum(S_s - threshold, 0)
    return U_s @ np.diag(S_thresholded) @ Vt_s

# ADMM for RPCA, with the standard parameter choices of
# Candes et al. (2011): lam = 1/sqrt(max(m,n)), mu from ||M||_1
lam = 1.0 / np.sqrt(max(m, n))
mu = m * n / (4 * np.sum(np.abs(M)))

L = np.zeros((m, n))
S = np.zeros((m, n))
Y = np.zeros((m, n))

for iteration in range(200):
    # Update L (nuclear norm minimization)
    L_temp = M - S + Y / mu
    L = svd_soft_threshold(L_temp, 1.0 / mu)

    # Update S (L1 minimization)
    S_temp = M - L + Y / mu
    S = soft_threshold(S_temp, lam / mu)

    # Update dual variable
    residual = M - L - S
    Y = Y + mu * residual

    # Check convergence (relative residual)
    if np.linalg.norm(residual, 'fro') < 1e-7 * np.linalg.norm(M, 'fro'):
        break

print(f"True rank: {rank}")
print(f"Recovered rank: {np.sum(np.linalg.svd(L)[1] > 1e-3)}")
print(f"Low-rank recovery error: {np.linalg.norm(L - L_true)/np.linalg.norm(L_true):.6f}")
print(f"Sparsity ratio (S): {np.sum(np.abs(S) > 1e-4) / (m*n):.4f}")

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Original data
axes[0, 0].imshow(M, cmap='gray')
axes[0, 0].set_title('Observed M = L + S')

axes[0, 1].imshow(L, cmap='gray')
axes[0, 1].set_title('Recovered Low-rank L')

axes[0, 2].imshow(S, cmap='gray')
axes[0, 2].set_title('Recovered Sparse S')

# Comparison
axes[1, 0].imshow(L_true, cmap='gray')
axes[1, 0].set_title('True L')

axes[1, 1].imshow(S_true, cmap='gray')
axes[1, 1].set_title('True S')

axes[1, 2].imshow(L + S, cmap='gray')
axes[1, 2].set_title('Reconstruction L + S')

for ax in axes.ravel():
    ax.axis('off')

plt.tight_layout()
plt.show()